# Additional aggregate checks: add `step6_additions.py` and run it
Runs the additional aggregate checks (cohort statuses, baseline table, grid summaries, smallest detectable difference, enrolment-order check, BP variability, exploratory coherence, affected hemisphere, environment) on the frozen Nov-4 run (`private/step4.pkl`).

**Run the cells top to bottom.** Output is saved to `reports/9_additions.txt` on Drive (aggregates only). Never share the `private/` folder.

## 1. Mount Drive and set the base folder

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, subprocess
BASE = __import__('os').environ.get('CVES_BASE', '/content/drive/MyDrive/CVES Fresh/CVES Repair')   # <- change if your folder holding kit/, private/, reports/ is elsewhere
assert os.path.exists(BASE + "/kit/core.py"), "BASE is wrong: no kit/core.py there"
os.environ["CVES_BASE"] = BASE
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "wfdb", "statsmodels"])
print("BASE ok:", BASE)

## 2. Load your helpers and restore the dataset
Copies the dataset from Drive to `/content/cves-1.0.0` if this is a fresh runtime (can take a while). Expect: head-up-tilt 68, sit-stand 71, sit-stand-balance 64, all OK.

In [ ]:
sys.path.insert(0, BASE + "/kit")
import importlib, helpers; importlib.reload(helpers)
cfg = helpers.setup()
helpers.check_data(cfg)
import numpy; print("numpy", numpy.__version__)

## 3. Write the new script and the reference output into your Drive folders
No manual upload needed. The script is the one stored in `kit/` of this repository; the reference output is stored in `reports/`.

In [ ]:
SCRIPT = r'''"""Additional aggregate checks: cohort statuses, grid summaries, smallest detectable difference, coherence by group, environment.
Plain: the few extra numbers the final paper needs, so none is computed by hand. Usage: step6_additions.py DATA step4.pkl. Aggregates only."""
import sys, re, platform, importlib, numpy as np, pandas as pd, statsmodels.formula.api as smf, warnings, wfdb
from scipy import signal as sg
from scipy.stats import spearmanr, mannwhitneyu
from sklearn.metrics import roc_auc_score
from core import records, pid, first_seated, beats_seg
from pathlib import Path
warnings.filterwarnings("ignore")
root = Path(sys.argv[1]); pk = Path(sys.argv[2]); G = pd.read_csv(pk.with_suffix(".grid.csv")); D = pd.read_pickle(pk)
s = pd.read_csv(root / "subjects.csv"); s.columns = [c.strip() for c in s.columns]; s["sid"] = s.subject_number.astype(str).str.strip().str.lower().str[:5]
print("== environment:", "python", platform.python_version(), *[f"{m} {importlib.import_module(m).__version__}" for m in ("numpy", "scipy", "pandas", "statsmodels", "sklearn", "wfdb")])
grp = s.group.astype(str).str.strip().str.upper()
print("\n== cohort: participant table rows", len(s), "| stroke", int((grp == "STROKE").sum()), "control", int((grp == "CONTROL").sum()))
print("visit status counts:", s.completed_visit_status.astype(str).str.strip().str.upper().value_counts().to_dict())
ids = {p: {h.stem[:5].lower() for h in (root / "data" / p).glob("*.hea")} for p in ("sit-stand-balance", "head-up-tilt")}
for nm, k in (("sit-stand-balance", ids["sit-stand-balance"]), ("head-up-tilt", ids["head-up-tilt"]), ("both", ids["sit-stand-balance"] & ids["head-up-tilt"]), ("either", ids["sit-stand-balance"] | ids["head-up-tilt"])):
    x = s[s.sid.isin(k)]; print(f"{nm}: n {len(x)}, stroke {int((x.group.str.upper().str.strip() == 'STROKE').sum())}, TIA {int(x['Stroke Group'].astype(str).str.strip().str.upper().eq('TIA').sum())}, completed {int(x.completed_visit_status.astype(str).str.upper().str.contains('COMPLET').sum())}")
from clinical import build
comp = s[s.completed_visit_status.astype(str).str.upper().str.contains("COMPLET") & s.sid.isin(ids["sit-stand-balance"])].set_index("sid"); X, yb = build(comp)
num = lambda c: pd.to_numeric(comp[c], errors="coerce")
X["nihss"] = num("NIHSS"); X["mrs"] = num("MRS"); X["sbp"] = num("SBP BASELINE"); X["dbp"] = num("DBP BASELINE"); X["hr"] = num("(Baseline Mean) HR BP BASELINE")
side = comp["Stroke Side"].astype(str).str.upper().str.strip()
print("\n== baseline table, sit-stand-balance cohort (stroke / control): median [IQR] or n (%); n with data")
for c in ("age", "bmi", "pack_years", "gait", "sbp", "dbp", "hr", "nihss", "mrs"):
    out = []
    for g in (1, 0):
        v = X[c][yb == g].dropna(); out.append(f"{v.median():.1f} [{v.quantile(.25):.1f}-{v.quantile(.75):.1f}] (n {len(v)})" if len(v) else "NA")
    print(f"  {c}: {out[0]} / {out[1]}")
for c in ("sex", "htn", "diabetes", "hyperlip", "cur_tob", "antiplatelet", "anticoag", "statin", "antihyperlip"):
    out = []
    for g in (1, 0):
        v = X[c][yb == g].dropna(); out.append(f"{int(v.sum())}/{len(v)} ({100 * v.mean():.0f}%)")
    print(f"  {c} (yes): {out[0]} / {out[1]}")
print("  stroke side among stroke:", side[yb == 1].map(lambda v: "bilateral" if "BILAT" in v else ("left" if v.startswith("L") else ("right" if v.startswith("R") else "not recorded or other"))).value_counts().to_dict())
print("\n== phase grid summaries (corrected clock)")
main = G[G.seg.isin(["first300", "t60-360"]) & (G.gate == "CARNet")]
for nm, g in (("12 main-gate cells, both sides, all arms", main), ("6 main-gate cells, left", main[main.sd == "L"]), ("8 main-gate aligned cells", main[main.arm.isin(["pr", "ro"])]),
              ("36 aligned cells, three segments x three gates", G[G.seg.isin(["first300", "t60-360", "longest300"]) & G.arm.isin(["pr", "ro"])])):
    print(f"{nm}: n cells {len(g)}, estimates {g.b.min():.1f} to {g.b.max():.1f}, median {g.b.median():.1f}, intervals excluding 0 {int(((g.lo > 0) | (g.hi < 0)).sum())}, median half-width {((g.hi - g.lo) / 2).median():.1f}, widest single interval {g.loc[(g.hi - g.lo).idxmax(), 'lo']:.1f} to {g.loc[(g.hi - g.lo).idxmax(), 'hi']:.1f}")
m = main.assign(mde=2.8 * (main.hi - main.lo) / (2 * 1.96))
print("smallest detectable difference (80 percent power, two-sided 5 percent; 2.8 x SE), main gate: median %.1f, range %.1f to %.1f deg" % (m.mde.median(), m.mde.min(), m.mde.max()))
for (p, sd), g in m.groupby(["proto", "sd"]): print(f"  {p} {sd}: {g.mde.min():.1f} to {g.mde.max():.1f}")
print("\n== enrolment-order proxy: participant number vs first recording date in dated headers (aggregate only)")
first = {}
for h in (root / "data").rglob("*.hea"):
    try: hd = wfdb.rdheader(str(h.with_suffix("")))
    except Exception: continue
    mm = re.match(r"s(\d{4})", h.name.lower())
    if hd.base_date and mm: k = int(mm.group(1)); first[k] = min(first.get(k, 10 ** 9), hd.base_date.toordinal())
ks = sorted(first); print("participants with a dated header", len(ks), "| Spearman %.3f" % spearmanr(ks, [first[i] for i in ks])[0])
kk = [i for i in ks if f"s{i:04d}" in set(s.sid)]; gg = [int(s.set_index("sid").group.astype(str).str.strip().str.upper()[f"s{i:04d}"] == "STROKE") for i in kk]; print("recording date predicts group: AUC %.3f (n %d)" % (roc_auc_score(gg, [first[i] for i in kk]), len(kk)))
print("\n== blood-pressure variability: ABP low-frequency power by group")
rows = []
for proto, fs in (("head-up-tilt", 500), ("sit-stand-balance", 1000)):
    for h in records(root, proto):
        r = wfdb.rdrecord(str(h.with_suffix(""))); n = r.sig_name; P = r.p_signal; q = first_seated(P[:, n.index("fz")], fs) if proto[0] == "s" else (60 * fs, 360 * fs)
        if q is None: continue
        a = np.nan_to_num(P[:, n.index("abp")]); x, _ = beats_seg(a, a, fs, *q); x = (x / x.mean() - 1) * 100; f, pw = sg.welch(x - x.mean(), fs=10, nperseg=1000, noverlap=500, window="hann"); mk_ = (f >= .07) & (f <= .2)
        rows.append(dict(proto=proto, sid=pid(h), lf=float(np.trapezoid(pw[mk_], f[mk_]))))
B = pd.DataFrame(rows); B["stroke"] = (B.sid.map(s.set_index("sid").group).astype(str).str.strip().str.upper() == "STROKE").astype(int)
for p, x in B.groupby("proto"): print(p, "median stroke %.2f vs control %.2f; Mann-Whitney p %.3f" % (x[x.stroke == 1].lf.median(), x[x.stroke == 0].lf.median(), mannwhitneyu(x[x.stroke == 1].lf, x[x.stroke == 0].lf).pvalue))
D = D.merge(B[["proto", "sid", "lf"]], on=["proto", "sid"], how="left"); D["llf"] = np.log(D.lf)
print("\n== EXPLORATORY: low-frequency coherence (0.07-0.20 Hz), stroke minus control, left, unaligned")
D["num"] = D.sid.str[1:].astype(int); S_ = s.set_index("sid")
for (p, seg), x in D[(D.sd == "L") & D.seg.isin(["first300", "second300", "t60-360"])].groupby(["proto", "seg"]):
    x = x.copy(); x["z"] = (x.num - x.num.mean()) / x.num.std(); x["age"] = pd.to_numeric(x.sid.map(S_.age), errors="coerce"); x["sx"] = x.sid.map(S_.gender).astype(str).str[0]
    q = x.dropna(subset=["un_coh", "age", "et", "llf"])
    for f in ("stroke + hi", "stroke + hi + z + age + C(sx) + et + map", "stroke + hi + z + llf"):
        r = smf.ols("un_coh ~ " + f, q).fit(cov_type="HC3", use_t=True); c = r.conf_int().loc["stroke"]; print(f"{p} {seg} | {f} | n {len(q)} | {r.params['stroke']:+.3f} ({c[0]:+.3f}, {c[1]:+.3f}) p {r.pvalues['stroke']:.3f}")
    rng = np.random.default_rng(7); obs = q[q.stroke == 1].un_coh.median() - q[q.stroke == 0].un_coh.median(); cnt = 0
    for _ in range(5000):
        y = q.stroke.values.copy()
        for h in (0, 1): i = np.where(q.hi.values == h)[0]; y[i] = rng.permutation(y[i])
        cnt += abs(np.median(q.un_coh.values[y == 1]) - np.median(q.un_coh.values[y == 0])) >= abs(obs)
    print(f"   median difference {obs:+.3f}; permutation within regime (5000) p {(cnt + 1) / 5001:.3f}")
    print("   medians by group x regime (n, coherence):", {("S" if a else "C") + ("hi" if b else "lo"): (len(y), round(y.un_coh.median(), 2)) for (a, b), y in q.groupby(["stroke", "hi"])})

print("\n== affected hemisphere (stroke, side recorded) vs controls (mean of sides), main gate, per-record aligned")
side = S_["Stroke Side"].astype(str).str.upper().str.strip()
aff = side.map(lambda v: None if "BILAT" in v else ("L" if re.match(r"L|LEFT", v) else ("R" if re.match(r"R|RIGHT", v) else None)))
for (p, seg), x in D[D.seg.isin(["first300", "t60-360"])].groupby(["proto", "seg"]):
    w = x.pivot_table(index="sid", values="pr_ph", columns="sd"); st = x.drop_duplicates("sid").set_index("sid").stroke; vals = []
    for sid, row in w.iterrows():
        if st[sid] == 1:
            a_ = aff.get(sid)
            if a_ in ("L", "R") and pd.notna(row.get(a_)): vals.append((sid, 1, row[a_]))
        elif row.notna().any(): vals.append((sid, 0, row.dropna().mean()))
    V = pd.DataFrame(vals, columns=["sid", "stroke", "ph"]); V["age"] = pd.to_numeric(V.sid.map(S_.age), errors="coerce"); r = smf.ols("ph ~ stroke + age", V.dropna()).fit(cov_type="HC3", use_t=True); c = r.conf_int().loc["stroke"]
    print(f"{p}: stroke {int((V.stroke == 1).sum())}, control {int((V.stroke == 0).sum())}; affected side minus control {r.params['stroke']:+.1f} ({c[0]:.1f}, {c[1]:.1f}) p {r.pvalues['stroke']:.2f}")
'''

(cfg.kit / 'step6_additions.py').write_text(SCRIPT)
print('wrote', cfg.kit / 'step6_additions.py', len(SCRIPT.splitlines()), 'lines')

In [ ]:
REFERENCE = r'''
== cohort: participant table rows 172 | stroke 84 control 88
visit status counts: {'COMPLETED': 91, 'EXCLUDED': 55, 'INELIGIBLE': 12, 'LOST TO FOLLOW-UP': 6, 'WITHDREW CONSENT': 4, 'V1': 3, 'EXCLUDED PENDING PCP': 1}
sit-stand-balance: n 64, stroke 37, TIA 1, completed 64
head-up-tilt: n 68, stroke 40, TIA 1, completed 68
both: n 63, stroke 36, TIA 1, completed 63
either: n 69, stroke 41, TIA 1, completed 69

== baseline table, sit-stand-balance cohort (stroke / control): median [IQR] or n (%); n with data
  age: 64.0 [55.0-71.0] (n 37) / 67.0 [62.5-71.0] (n 27)
  bmi: 26.9 [24.7-29.2] (n 37) / 24.4 [21.9-27.7] (n 27)
  pack_years: 19.5 [0.4-46.8] (n 30) / 0.0 [0.0-9.0] (n 25)
  gait: 0.9 [0.8-1.0] (n 36) / 1.2 [1.1-1.2] (n 27)
  sbp: 125.8 [119.8-140.0] (n 37) / 124.3 [109.2-134.9] (n 27)
  dbp: 58.7 [53.9-67.9] (n 37) / 60.3 [52.0-64.1] (n 27)
  hr: 68.8 [64.7-72.2] (n 37) / 62.7 [56.8-67.0] (n 27)
  nihss: 2.0 [1.0-3.0] (n 37) / NA
  mrs: 1.0 [0.0-2.0] (n 36) / NA
  sex (yes): 18/37 (49%) / 12/27 (44%)
  htn (yes): 26/37 (70%) / 13/27 (48%)
  diabetes (yes): 1/37 (3%) / 1/27 (4%)
  hyperlip (yes): 25/37 (68%) / 11/27 (41%)
  cur_tob (yes): 8/35 (23%) / 1/27 (4%)
  antiplatelet (yes): 28/37 (76%) / 7/27 (26%)
  anticoag (yes): 5/37 (14%) / 0/27 (0%)
  statin (yes): 25/37 (68%) / 4/27 (15%)
  antihyperlip (yes): 5/37 (14%) / 1/27 (4%)
  stroke side among stroke: {'right': 17, 'left': 16, 'not recorded or other': 3, 'bilateral': 1}

== phase grid summaries (corrected clock)
12 main-gate cells, both sides, all arms: n cells 12, estimates -3.3 to 3.3, median -0.4, intervals excluding 0 0, median half-width 12.4, widest single interval -24.1 to 17.6
6 main-gate cells, left: n cells 6, estimates -3.3 to 0.1, median -1.4, intervals excluding 0 0, median half-width 15.1, widest single interval -24.1 to 17.6
8 main-gate aligned cells: n cells 8, estimates -3.3 to 3.3, median 0.2, intervals excluding 0 0, median half-width 12.2, widest single interval -24.1 to 17.6
36 aligned cells, three segments x three gates: n cells 36, estimates -8.0 to 10.3, median -1.0, intervals excluding 0 0, median half-width 13.2, widest single interval -24.1 to 17.6
smallest detectable difference (80 percent power, two-sided 5 percent; 2.8 x SE), main gate: median 17.7, range 13.9 to 29.7 deg
  head L: 17.2 to 18.7
  head R: 13.9 to 15.4
  sit- L: 24.4 to 29.7
  sit- R: 17.0 to 18.6

== enrolment-order proxy: participant number vs first recording date in dated headers (aggregate only)
participants with a dated header 91 | Spearman 0.959
recording date predicts group: AUC 0.853 (n 90)

== blood-pressure variability: ABP low-frequency power by group
head-up-tilt median stroke 2.66 vs control 2.07; Mann-Whitney p 1.000
sit-stand-balance median stroke 1.84 vs control 1.83; Mann-Whitney p 0.796

== EXPLORATORY: low-frequency coherence (0.07-0.20 Hz), stroke minus control, left, unaligned
head-up-tilt t60-360 | stroke + hi | n 62 | -0.149 (-0.286, -0.012) p 0.033
head-up-tilt t60-360 | stroke + hi + z + age + C(sx) + et + map | n 62 | -0.168 (-0.307, -0.030) p 0.018
head-up-tilt t60-360 | stroke + hi + z + llf | n 62 | -0.176 (-0.310, -0.041) p 0.011
   median difference -0.120; permutation within regime (5000) p 0.548
   medians by group x regime (n, coherence): {'Clo': (20, 0.41), 'Chi': (5, 0.79), 'Slo': (11, 0.29), 'Shi': (26, 0.45)}
sit-stand-balance first300 | stroke + hi | n 60 | -0.163 (-0.304, -0.023) p 0.024
sit-stand-balance first300 | stroke + hi + z + age + C(sx) + et + map | n 60 | -0.199 (-0.346, -0.052) p 0.009
sit-stand-balance first300 | stroke + hi + z + llf | n 60 | -0.204 (-0.339, -0.069) p 0.004
   median difference -0.249; permutation within regime (5000) p 0.008
   medians by group x regime (n, coherence): {'Clo': (19, 0.58), 'Chi': (7, 0.64), 'Slo': (11, 0.32), 'Shi': (23, 0.37)}
sit-stand-balance second300 | stroke + hi | n 58 | -0.138 (-0.275, -0.001) p 0.049
sit-stand-balance second300 | stroke + hi + z + age + C(sx) + et + map | n 58 | -0.180 (-0.325, -0.035) p 0.016
sit-stand-balance second300 | stroke + hi + z + llf | n 58 | -0.186 (-0.313, -0.058) p 0.005
   median difference -0.121; permutation within regime (5000) p 0.115
   medians by group x regime (n, coherence): {'Clo': (19, 0.5), 'Chi': (6, 0.73), 'Slo': (11, 0.38), 'Shi': (22, 0.44)}

== affected hemisphere (stroke, side recorded) vs controls (mean of sides), main gate, per-record aligned
head-up-tilt: stroke 32, control 25; affected side minus control -2.6 (-13.5, 8.2) p 0.63
sit-stand-balance: stroke 23, control 25; affected side minus control -8.4 (-23.1, 6.3) p 0.26
'''

cfg.ref.mkdir(parents=True, exist_ok=True)
(cfg.ref / '9_additions.txt').write_text(REFERENCE)
print('wrote', cfg.ref / '9_additions.txt', len(REFERENCE.splitlines()), 'lines')

## 4. Check the inputs exist
All four must print `True`.

In [ ]:
for f in (cfg.kit/"step6_additions.py", cfg.work/"step4.pkl", cfg.work/"step4.grid.csv", cfg.ref/"9_additions.txt"):
    print(f.exists(), f)

## 5. NumPy compatibility
The script uses `np.trapezoid` (NumPy >= 2.0). This patches it only if your NumPy is older.

In [ ]:
p = cfg.kit/"step6_additions.py"; t = p.read_text()
if int(numpy.__version__.split(".")[0]) < 2 and "np.trapezoid" in t:
    p.write_text(t.replace("np.trapezoid", "np.trapz")); print("patched for NumPy 1.x")
else:
    print("no patch needed")

## 6. Run the step
Takes a few minutes (reads every record). Saves `reports/9_additions.txt`.

In [ ]:
txt = helpers.run_step(cfg, "9_additions", "step6_additions.py", cfg.data, cfg.work/"step4.pkl")

In [ ]:
p = cfg.kit/"run_all.sh"; t = p.read_text()
line = 'step 9_additions $PY step6_additions.py "$R" "$W/step4.pkl"\n'
marker = 'echo "All steps passed'
if "9_additions" not in t and marker in t:
    p.write_text(t.replace(marker, line + marker)); print("added")
else:
    print("already there or marker not found")

In [ ]:
import sys, pandas as pd
sys.path.insert(0, str(cfg.kit)); from clinical import build
s = pd.read_csv(cfg.data/"subjects.csv"); s.columns = [c.strip() for c in s.columns]
s["sid"] = s.subject_number.astype(str).str.strip().str.lower().str[:5]
ids = {h.stem[:5].lower() for h in (cfg.data/"data"/"sit-stand-balance").glob("*.hea")}
comp = s[s.completed_visit_status.astype(str).str.upper().str.contains("COMPLET") & s.sid.isin(ids)].set_index("sid")
X, y = build(comp)
for g, n in ((1, "stroke"), (0, "control")):
    v = X.gait[y == g].dropna(); print(n, f"{v.median():.2f} [{v.quantile(.25):.2f}-{v.quantile(.75):.2f}] n={len(v)}")

## 7. Compare with the reference run
Shows only lines that differ or are missing. Counts must match exactly; phase numbers within about 1 degree. The `== environment:` line exists only in your run.

In [ ]:
helpers.compare(cfg, "9_additions", show_ok=False)

## 8. Print the report in blocks (copy these back to Claude)

In [ ]:
import re
rep = helpers.read_report(cfg, "9_additions")
for key in ("environment", "cohort", "baseline", "phase grid", "enrolment", "blood-pressure", "EXPLORATORY", "affected"):
    print(helpers.section(rep, key) or "[no block starting with '== %s']" % key); print()

In [ ]:
# Fallback if the block above prints nothing: print the whole file
print(rep)

In [ ]:
import subprocess
print(subprocess.run("ls -R /content/cves-1.0.0/conversion | head -40; "
  "grep -rIn -E '1000|500|fs|sampling' /content/cves-1.0.0/conversion --include='*.py' --include='*.ipynb' --include='*.csv' --include='*.txt' | head -40",
  shell=True, capture_output=True, text=True).stdout)

In [ ]:
import subprocess
print(subprocess.run("ls /content/cves-1.0.0 | head -30; "
  "grep -rIl -E '1000 ?Hz|1000' /content/cves-1.0.0 --include='*.csv' --include='*.txt' --include='*.md' --include='*.html' | head; "
  "grep -rIn -iE 'sit.?stand.?balance.*(1000|500)|(1000|500).*sit.?stand.?balance' /content/cves-1.0.0 --include='*.csv' --include='*.txt' --include='*.md' --include='*.html' | head -20",
  shell=True, capture_output=True, text=True).stdout)